# Full Music Generation with Real EnCodec (ALD-SC)

This notebook demonstrates the **fully-fledged ALD-SC music-generation pipeline** using the real frozen EnCodec encoder instead of the stub encoder used in the earlier notebooks.

## Pipeline
1. Build a synthetic music corpus with `MusicSynthDataset` (harmonic tones, ADSR envelopes, tremolo, noise bursts).
2. Extract real EnCodec latents and build the frozen ArrowSpace prior.
3. Train a graph decoder vs. a matched-capacity baseline decoder.
4. Train a 1-D DiT denoiser on the real EnCodec latent space.
5. Sample, decode, and evaluate.

> LSD has no research claim — it's a sound-production tool. The goal is simply to generate novel sounds and explore *the sound of the future*.

In [1]:
# --- Generation knobs ---
SEED = 3407
STEPS = 50
TEMPERATURE = 0.85
USE_C_SPEC = True

# --- Audio / dataset knobs ---
AUDIO_SECONDS = 2.0          # length of each training clip
NUM_SAMPLES = 32             # size of synthetic music corpus
SAMPLE_RATE = 24000

# --- Model knobs ---
Q = 8                        # prior chart dimension
K = 4                        # prior knn
BASE_CHANNELS = 32

# --- Training knobs ---
DECODER_EPOCHS = 20
DIFFUSION_EPOCHS = 20
BATCH_SIZE = 4
LR = 1e-3

AUDIO_LENGTH = int(AUDIO_SECONDS * SAMPLE_RATE)
LATENT_LENGTH = AUDIO_LENGTH // 320  # EnCodec 24kHz stride
print(f'Audio: {AUDIO_SECONDS}s ({AUDIO_LENGTH} samples) -> latent length {LATENT_LENGTH}')
print(f'Corpus: {NUM_SAMPLES} clips, prior q={Q}, k={K}')
print(f'Knobs: seed={SEED}, steps={STEPS}, temp={TEMPERATURE}, use_c_spec={USE_C_SPEC}')

Audio: 2.0s (48000 samples) -> latent length 150
Corpus: 32 clips, prior q=8, k=4
Knobs: seed=3407, steps=50, temp=0.85, use_c_spec=True


## Imports

In [2]:
import torch
import torch.nn as nn
import torchaudio
from IPython.display import Audio, display

from ald_sc.build_prior import build_arrow_prior
from ald_sc.audio_codec import EnCodecEncoder, AudioVAE, BaselineAudioDecoder
from ald_sc.graph_decoder import GraphDecoder
from ald_sc.dit import MinimalDiT
from ald_sc.data import MusicSynthDataset, build_audio_dataloader
from ald_sc.losses import ALDSCLoss
from ald_sc.schedule import CosineSchedule
from ald_sc.sampling import sample_ddim
from ald_sc.trainer import train_audio_decoder, train_audio_diffusion, log_training

device = torch.device('cpu')
torch.manual_seed(SEED)
print('Imports done. Device:', device)
print('EnCodecEncoder loaded lazily — first encode call may download weights.')

Imports done. Device: cpu
EnCodecEncoder loaded lazily — first encode call may download weights.


## Step 1: Build the ArrowSpace Prior from Real EnCodec Features

In [3]:
# Use the real frozen EnCodec encoder (no stub)
encoder = EnCodecEncoder(sample_rate=SAMPLE_RATE, bandwidth=24)

# Synthetic music corpus
dataset = MusicSynthDataset(
    num_samples=NUM_SAMPLES,
    audio_length=AUDIO_LENGTH,
    sample_rate=SAMPLE_RATE,
    seed=SEED,
    num_harmonics=4,
)
loader = build_audio_dataloader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# Extract pooled EnCodec features for the prior
features = []
for batch in loader:
    z = encoder.extract_features(batch)
    features.append(z.mean(dim=2))
embeddings = torch.cat(features, dim=0)
print(f'Corpus EnCodec embeddings: {embeddings.shape}')

# Build the frozen ArrowSpace prior
prior = build_arrow_prior(embeddings, q=Q, k=K)
print(f'L_F: {prior.L_F.shape}, U_q: {prior.U_q.shape}, q={prior.q}')

Corpus EnCodec embeddings: torch.Size([32, 128])
L_F: torch.Size([128, 128]), U_q: torch.Size([128, 8]), q=8


## Step 2: Train Graph Decoder vs. Baseline Decoder

In [4]:
# Graph decoder (uses real EnCodec latents + ArrowSpace prior)
graph_decoder = GraphDecoder(
    latent_channels=128,
    out_channels=1,
    feature_dim=128,
    base_channels=BASE_CHANNELS,
    prior=prior,
    upsample_strides=(2, 4, 5, 8),
)

# Matched-capacity baseline decoder (no graph structure)
baseline_decoder = BaselineAudioDecoder(
    latent_channels=128,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    upsample_strides=(2, 4, 5, 8),
)

loss_fn = ALDSCLoss(
    prior=prior,
    lambda_rec=1.0,
    lambda_stft=0.0,
    lambda_chart=0.5,
    lambda_smooth=0.1,
)
train_loader = build_audio_dataloader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Train graph decoder
graph_vae = AudioVAE(encoder=encoder, decoder=graph_decoder)
print('Training graph decoder on real EnCodec latents...')
graph_losses = list(train_audio_decoder(
    train_loader, graph_vae, prior, loss_fn,
    epochs=DECODER_EPOCHS, lr=LR, device=device,
))
print(f'  Loss: {graph_losses[0]["loss"]:.4f} -> {graph_losses[-1]["loss"]:.4f}')

# Train baseline decoder
baseline_vae = AudioVAE(encoder=encoder, decoder=baseline_decoder)
print('Training baseline decoder...')
baseline_losses = list(train_audio_decoder(
    train_loader, baseline_vae, prior, loss_fn,
    epochs=DECODER_EPOCHS, lr=LR, device=device,
))
print(f'  Loss: {baseline_losses[0]["loss"]:.4f} -> {baseline_losses[-1]["loss"]:.4f}')

Training graph decoder on real EnCodec latents...
  Loss: 0.3709 -> 0.2748
Training baseline decoder...
  Loss: 0.4020 -> 0.2176


## Step 3: Train the 1-D DiT Denoiser

In [5]:
dit = MinimalDiT(
    latent_channels=128,
    latent_length=LATENT_LENGTH,
    patch_size=8,
    dim=64,
    depth=2,
    num_heads=4,
    spec_dim=3 * Q,
)
sched = CosineSchedule(num_steps=1000)

# Freeze VAE for diffusion training
for p in graph_vae.parameters():
    p.requires_grad_(False)

print(f'Training 1-D DiT on real EnCodec latents (latent_length={LATENT_LENGTH})...')
diff_losses = list(train_audio_diffusion(
    train_loader, graph_vae, dit, prior, sched,
    epochs=DIFFUSION_EPOCHS, lr=LR, device=device,
))
print(f'  Loss: {diff_losses[0]["loss"]:.4f} -> {diff_losses[-1]["loss"]:.4f}')

Training 1-D DiT on real EnCodec latents (latent_length=150)...
  Loss: 14.3222 -> 6.1122


## Step 4: Generate Music

In [6]:
# Sample latent z from noise
torch.manual_seed(SEED)
dit = dit.eval()
z = sample_ddim(dit, sched, batch_size=1, steps=STEPS, seed=SEED, device=device)

# Apply temperature scaling
z = z * TEMPERATURE
print(f'Sampled z: {z.shape}')

# Derive c_spec from z (self-consistent decoding)
a = z.mean(dim=2)
c_spec = prior.chart_energy_descriptor(a)

# Decode with graph decoder
with torch.no_grad():
    if USE_C_SPEC:
        audio_graph = graph_decoder(z, c_spec)
    else:
        audio_graph = graph_decoder(z, torch.zeros_like(c_spec))
    audio_baseline = baseline_decoder(z)

# Normalize for playback
def normalize(audio):
    audio = audio.squeeze(0)
    audio = audio - audio.mean()  # DC-block: decoder emits large DC (PR #59)
    peak = audio.abs().max()
    if peak > 0:
        audio = audio / peak
    return audio

audio_graph_norm = normalize(audio_graph)
audio_baseline_norm = normalize(audio_baseline)

print(f'Graph decoder audio: {audio_graph_norm.shape}, {audio_graph_norm.shape[-1]/SAMPLE_RATE:.2f}s')
print(f'Baseline decoder audio: {audio_baseline_norm.shape}, {audio_baseline_norm.shape[-1]/SAMPLE_RATE:.2f}s')

Sampled z: torch.Size([1, 128, 150])
Graph decoder audio: torch.Size([1, 48000]), 2.00s
Baseline decoder audio: torch.Size([1, 48000]), 2.00s


In [7]:
print('Graph decoder output:')
display(Audio(audio_graph_norm.numpy(), rate=SAMPLE_RATE))

Graph decoder output:


In [8]:
print('Baseline decoder output:')
display(Audio(audio_baseline_norm.numpy(), rate=SAMPLE_RATE))

Baseline decoder output:


In [9]:
print('Real music-synth training clip (reference):')
real_clip = dataset[0]  # (1, T)
display(Audio(real_clip.numpy(), rate=SAMPLE_RATE))

Real music-synth training clip (reference):


## Step 5: Evaluation

In [10]:
eval_loader = build_audio_dataloader(dataset, batch_size=BATCH_SIZE, shuffle=False)

def eval_reconstruction(vae, loader, loss_fn, device):
    vae.eval()
    total_rec, total_chart, n = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            x = batch.to(device)
            z, A, c_spec, x_hat = vae(x, prior)
            losses = loss_fn(x, x_hat, A, A.detach())
            total_rec += losses['rec'].item()
            total_chart += losses['chart'].item()
            n += 1
    return {'rec': total_rec/n, 'chart': total_chart/n}

graph_metrics = eval_reconstruction(graph_vae, eval_loader, loss_fn, device)
baseline_metrics = eval_reconstruction(baseline_vae, eval_loader, loss_fn, device)

print('=== Reconstruction Comparison (real EnCodec latents) ===')
print(f'Graph decoder:    L1={graph_metrics["rec"]:.6f}  chart={graph_metrics["chart"]:.6f}')
print(f'Baseline decoder: L1={baseline_metrics["rec"]:.6f}  chart={baseline_metrics["chart"]:.6f}')
diff = baseline_metrics['rec'] - graph_metrics['rec']
print(f'Graph improvement: {diff:+.6f} (positive = graph is better)')

=== Reconstruction Comparison (real EnCodec latents) ===
Graph decoder:    L1=0.206639  chart=0.000000
Baseline decoder: L1=0.182621  chart=0.000000
Graph improvement: -0.024018 (positive = graph is better)


## Step 6: lambda_ED Ablation

In [11]:
class NoCSPecVAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x, prior):
        z, a, c_spec = self.encoder.encode(x, prior)
        zero_cspec = torch.zeros_like(c_spec)
        x_hat = self.decoder(z, zero_cspec)
        return z, a, c_spec, x_hat

ablation_vae = NoCSPecVAE(encoder, graph_decoder)
ablation_metrics = eval_reconstruction(ablation_vae, eval_loader, loss_fn, device)

print('=== lambda_ED Ablation ===')
print(f'With c_spec:    L1={graph_metrics["rec"]:.6f}  chart={graph_metrics["chart"]:.6f}')
print(f'Without c_spec: L1={ablation_metrics["rec"]:.6f}  chart={ablation_metrics["chart"]:.6f}')
diff = ablation_metrics['rec'] - graph_metrics['rec']
print(f'lambda_ED effect: {diff:+.6f} (positive = gating helps)')

=== lambda_ED Ablation ===
With c_spec:    L1=0.206639  chart=0.000000
Without c_spec: L1=0.206555  chart=0.000000
lambda_ED effect: -0.000084 (positive = gating helps)


## Summary

In [12]:
print('=== Summary ===')
print(f'Encoder: real EnCodec 24kHz (frozen)')
print(f'Dataset: MusicSynthDataset ({NUM_SAMPLES} clips, {AUDIO_SECONDS}s each)')
print(f'Prior: ArrowSpace q={Q}, k={K}')
print(f'Graph decoder loss: {graph_losses[-1]["loss"]:.4f}')
print(f'Baseline decoder loss: {baseline_losses[-1]["loss"]:.4f}')
print(f'DiT loss: {diff_losses[-1]["loss"]:.4f}')
print('\nRe-run the knob cell with different SEED, STEPS, TEMPERATURE, or')
print('USE_C_SPEC to explore the generated music space.')

=== Summary ===
Encoder: real EnCodec 24kHz (frozen)
Dataset: MusicSynthDataset (32 clips, 2.0s each)
Prior: ArrowSpace q=8, k=4
Graph decoder loss: 0.2748
Baseline decoder loss: 0.2176
DiT loss: 6.1122

Re-run the knob cell with different SEED, STEPS, TEMPERATURE, or
USE_C_SPEC to explore the generated music space.
